# Coordinate-Wise Length Smoke Test

This notebook checks the reviewer-requested coordinate-wise region length outputs. Run it from the repository root.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "utility").exists():
    raise RuntimeError(f"Run this notebook from the repository root, not {repo_root}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

In [ ]:
from utility.exps import run_abs_res_synthetic_experiment

methods = ["TSCP_R", "Empirical_copula", "Unscaled", "Point_CHR"]

result = run_abs_res_synthetic_experiment(
    dim_list=[4],
    sample_list=[30],
    alpha_list=[0.2],
    noise_type="Gaussian",
    trials=200,
    methods=methods,
    n_train=int(0.8 * (8000)), n_test=(8000) - int(0.8 * (8000)),
    n_features=6,
    n_informative=6,
    oracle_n_samples=200,
)

result.summary_results

In [ ]:
coord_trial = result.coordinate_trial_results
coord_summary = result.coordinate_summary_results

assert not coord_trial.empty
assert not coord_summary.empty
assert set(coord_summary["method"]) == set(methods)
assert set(coord_summary["coordinate"]) == {1, 2, 3, 4}
assert (coord_trial["coordinate_length"] >= 0).all()

expected_trial_rows = 1 * 1 * 1 * 200 * len(methods) * 4
assert len(coord_trial) == expected_trial_rows

coord_summary.sort_values(["method", "coordinate"])

In [ ]:
wide_lengths = coord_summary.pivot_table(
    index="coordinate",
    columns="method",
    values="coordinate_length_avg",
)

wide_lengths

Interpretation check: in the default synthetic setup, coordinate noise levels are `[d, d-1, ..., 1]`. A coordinate-adaptive method should usually show larger average lengths for noisier early coordinates and smaller lengths for quieter later coordinates, while less adaptive methods may look flatter across coordinates.